# Política de Decisão

Após a comparação dos algoritmos de aprendizado de máquina,
o CatBoost foi selecionado como modelo preditivo para as etapas
seguintes do projeto.

O modelo produz, para cada transação, um score contínuo associado
ao risco de fraude. Entretanto, esse score, isoladamente, ainda não
representa uma decisão operacional.

O objetivo desta etapa é transformar o score produzido pelo modelo
em três possíveis ações:

- **APROVAR**: transações com baixo risco estimado;
- **REVISAR**: transações com risco intermediário, encaminhadas para
  análise;
- **ALERTA_CRÍTICO**: transações com risco elevado, tratadas com maior
  prioridade.

A definição dos pontos de corte será realizada exclusivamente sobre
o conjunto de validação. O conjunto de teste permanecerá intocado
até que o modelo e a política de decisão estejam completamente
congelados.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(
    0,
    str(PROJECT_ROOT)
)

print(
    "Project root:",
    PROJECT_ROOT
)

Project root: /Users/lucassantos/Documents/ccard_fraud_ml


In [3]:
from src.inference import FraudModel

In [4]:
modelo_fraude = FraudModel()

In [5]:
print(
    "Árvores carregadas:",
    modelo_fraude.modelo.tree_count_
)

Árvores carregadas: 719


In [6]:
DATA_PATH = (
    PROJECT_ROOT
    / "raw"
    / "fraudTrain.csv"
)

In [7]:
dtypes = {
    "cc_num": "string",
    "trans_num": "string",
    "zip": "string",
    "merchant": "string",
    "category": "category",
    "gender": "category",
    "state": "category",
    "first": "string",
    "last": "string",
    "street": "string",
    "city": "string",
    "job": "string"
}

In [8]:
df = pd.read_csv(
    DATA_PATH,
    dtype=dtypes,
    parse_dates=[
        "trans_date_trans_time",
        "dob"
    ]
)

In [9]:
colunas_indice = [
    coluna
    for coluna in df.columns
    if str(coluna).startswith(
        "Unnamed:"
    )
]

if colunas_indice:
    df = df.drop(
        columns=colunas_indice
    )

In [10]:
print(
    "Transações:",
    f"{len(df):,}"
)

print(
    "Fraudes:",
    f"{df['is_fraud'].sum():,}"
)

Transações: 1,296,675
Fraudes: 7,506


In [11]:
df = (
    df
    .sort_values(
        "trans_date_trans_time"
    )
    .reset_index(drop=True)
)

In [12]:
print(
    "Primeira transação:",
    df[
        "trans_date_trans_time"
    ].min()
)

print(
    "Última transação:",
    df[
        "trans_date_trans_time"
    ].max()
)

Primeira transação: 2019-01-01 00:00:18
Última transação: 2020-06-21 12:13:37


In [13]:
n_total = len(df)

fim_treino = int(
    n_total * 0.70
)

fim_validacao = int(
    n_total * 0.85
)

In [14]:
df_treino = (
    df.iloc[
        :fim_treino
    ]
    .copy()
)

df_validacao = (
    df.iloc[
        fim_treino:fim_validacao
    ]
    .copy()
)

df_teste = (
    df.iloc[
        fim_validacao:
    ]
    .copy()
)

In [15]:
print(
    "Treino:",
    f"{len(df_treino):,}"
)

print(
    "Validação:",
    f"{len(df_validacao):,}"
)

print(
    "Teste:",
    f"{len(df_teste):,}"
)

Treino: 907,672
Validação: 194,501
Teste: 194,502


In [16]:
print(
    "Fraudes na validação:",
    f"{df_validacao['is_fraud'].sum():,}"
)

print(
    "Legítimas na validação:",
    f"{(df_validacao['is_fraud'] == 0).sum():,}"
)

Fraudes na validação: 1,252
Legítimas na validação: 193,249


In [17]:
assert len(df_validacao) == 194_501

assert (
    df_validacao[
        "is_fraud"
    ].sum()
    == 1_252
)

print(
    "Split de validação: OK"
)

Split de validação: OK


In [18]:
y_validacao = (
    df_validacao[
        "is_fraud"
    ]
    .to_numpy()
)

In [19]:
scores_validacao = (
    modelo_fraude.prever_scores(
        df_validacao
    )
)

In [20]:
print(
    "Scores gerados:",
    len(scores_validacao)
)

print(
    "Menor score:",
    scores_validacao.min()
)

print(
    "Maior score:",
    scores_validacao.max()
)

print(
    "Score médio:",
    scores_validacao.mean()
)

Scores gerados: 194501
Menor score: 7.583238277940751e-08
Maior score: 0.9999194434778264
Score médio: 0.0060069249679700065


In [21]:
assert (
    len(scores_validacao)
    ==
    len(df_validacao)
)

assert np.isfinite(
    scores_validacao
).all()

assert (
    scores_validacao >= 0
).all()

assert (
    scores_validacao <= 1
).all()

print(
    "Scores da validação: OK"
)

Scores da validação: OK


In [22]:
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

In [23]:
def avaliar_modelo(
    y_real,
    scores,
    threshold: float = 0.50
) -> pd.DataFrame:

    y_real = np.asarray(
        y_real
    ).ravel()

    scores = np.asarray(
        scores
    ).ravel()

    if len(y_real) != len(scores):
        raise ValueError(
            "y_real e scores precisam "
            "possuir a mesma quantidade "
            "de observações."
        )

    if not 0 <= threshold <= 1:
        raise ValueError(
            "O threshold deve estar "
            "entre 0 e 1."
        )

    y_previsto = (
        scores >= threshold
    ).astype(int)

    matriz = confusion_matrix(
        y_real,
        y_previsto,
        labels=[0, 1]
    )

    tn, fp, fn, tp = (
        matriz.ravel()
    )

    resultado = {
        "threshold":
            float(threshold),

        "average_precision":
            average_precision_score(
                y_real,
                scores
            ),

        "roc_auc":
            roc_auc_score(
                y_real,
                scores
            ),

        "accuracy":
            accuracy_score(
                y_real,
                y_previsto
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_real,
                y_previsto
            ),

        "precision_fraude":
            precision_score(
                y_real,
                y_previsto,
                pos_label=1,
                zero_division=0
            ),

        "recall_fraude":
            recall_score(
                y_real,
                y_previsto,
                pos_label=1,
                zero_division=0
            ),

        "f1_fraude":
            f1_score(
                y_real,
                y_previsto,
                pos_label=1,
                zero_division=0
            ),

        "verdadeiros_negativos":
            int(tn),

        "falsos_positivos":
            int(fp),

        "falsos_negativos":
            int(fn),

        "verdadeiros_positivos":
            int(tp),

        "percentual_alertado":
            y_previsto.mean() * 100
    }

    return pd.DataFrame(
        [resultado]
    )

In [24]:
resultado_validacao_050 = (
    avaliar_modelo(
        y_real=y_validacao,
        scores=scores_validacao,
        threshold=0.50
    )
)

In [25]:
resultado_validacao_050.T

,0
threshold,0.500000
average_precision,0.931645
roc_auc,0.996922
accuracy,0.998442
balanced_accuracy,0.892881
precision_fraude,0.965653
recall_fraude,0.785942
f1_fraude,0.866579
verdadeiros_negativos,193214.000000
falsos_positivos,35.000000


In [26]:
PASTA_TABELAS = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
    / "06_politica_decisao"
)

PASTA_TABELAS.mkdir(
    parents=True,
    exist_ok=True
)

In [27]:
validacao_scores = pd.DataFrame({
    "trans_num":
        df_validacao[
            "trans_num"
        ].to_numpy(),

    "trans_date_trans_time":
        df_validacao[
            "trans_date_trans_time"
        ].to_numpy(),

    "is_fraud":
        y_validacao,

    "score_fraude":
        scores_validacao
})

In [28]:
validacao_scores.head()

,trans_num,trans_date_trans_time,is_fraud,score_fraude
0,ab0a5c3d9d19f450f247bff3723c8a11,2019-12-28 17:18:38,0,7.583238e-08
1,bdbce4b44e706f543419b4a0ae40517d,2019-12-28 17:18:38,0,1.973227e-06
2,512641651a59d28d67c600070be091a8,2019-12-28 17:18:43,0,1.356613e-04
3,7c12870c6380fd8d8109e67e602fdf68,2019-12-28 17:18:46,0,5.342556e-05
4,020683600ccb9db996cd2db9cc779244,2019-12-28 17:18:48,0,4.136688e-05


In [29]:
CAMINHO_SCORES = (
    PASTA_TABELAS
    / "scores_validacao_catboost_v1.csv"
)

In [30]:
validacao_scores.to_csv(
    CAMINHO_SCORES,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Scores salvos em:",
    CAMINHO_SCORES.resolve()
)

Scores salvos em: /Users/lucassantos/Documents/ccard_fraud_ml/outputs/tables/06_politica_decisao/scores_validacao_catboost_v1.csv


## Análise dos thresholds

O CatBoost fornece um score contínuo associado ao risco de fraude.
Para transformar esse score em uma decisão operacional, é necessário
avaliar como diferentes pontos de corte afetam a quantidade de fraudes
detectadas, os falsos positivos e o volume de transações encaminhadas
para análise.

Nesta etapa, os thresholds são avaliados exclusivamente no conjunto
de validação. Nenhum ponto de corte é selecionado ainda.

In [31]:
scores = np.asarray(
    scores_validacao,
    dtype=float
)

y = np.asarray(
    y_validacao,
    dtype=int
)

In [32]:
ordem = np.argsort(
    -scores
)

scores_ordenados = scores[
    ordem
]

y_ordenado = y[
    ordem
]

In [33]:
tp_acumulado = np.cumsum(
    y_ordenado == 1
)

fp_acumulado = np.cumsum(
    y_ordenado == 0
)

In [34]:
total_fraudes = int(
    (y == 1).sum()
)

total_legitimas = int(
    (y == 0).sum()
)

total_transacoes = len(y)

In [35]:
fim_de_score = np.r_[
    scores_ordenados[:-1]
    !=
    scores_ordenados[1:],
    True
]

In [36]:
thresholds = (
    scores_ordenados[
        fim_de_score
    ]
)

tp = (
    tp_acumulado[
        fim_de_score
    ]
)

fp = (
    fp_acumulado[
        fim_de_score
    ]
)

fn = total_fraudes - tp
tn = total_legitimas - fp

alertas = tp + fp

In [37]:
precision = np.divide(
    tp,
    alertas,
    out=np.zeros_like(
        tp,
        dtype=float
    ),
    where=alertas != 0
)

recall = (
    tp
    /
    total_fraudes
)

f1 = np.divide(
    2 * precision * recall,
    precision + recall,
    out=np.zeros_like(
        precision,
        dtype=float
    ),
    where=(
        precision + recall
    ) != 0
)

percentual_alertado = (
    alertas
    /
    total_transacoes
    *
    100
)

In [38]:
tabela_thresholds = pd.DataFrame({
    "threshold": thresholds,
    "alertas": alertas,
    "percentual_alertado": percentual_alertado,

    "precision": precision,
    "recall": recall,
    "f1": f1,

    "verdadeiros_positivos": tp,
    "falsos_positivos": fp,
    "falsos_negativos": fn,
    "verdadeiros_negativos": tn
})

In [39]:
tabela_thresholds.head()

,threshold,alertas,percentual_alertado,precision,recall,f1,verdadeiros_positivos,falsos_positivos,falsos_negativos,verdadeiros_negativos
0,0.999919,1,0.000514,1.0,0.000799,0.001596,1,0,1251,193249
1,0.999838,2,0.001028,1.0,0.001597,0.003190,2,0,1250,193249
2,0.999833,3,0.001542,1.0,0.002396,0.004781,3,0,1249,193249
3,0.999829,4,0.002057,1.0,0.003195,0.006369,4,0,1248,193249
4,0.999813,5,0.002571,1.0,0.003994,0.007955,5,0,1247,193249


In [40]:
tabela_thresholds = (
    tabela_thresholds
    .sort_values(
        "threshold",
        ascending=False
    )
    .reset_index(drop=True)
)

In [41]:
tabela_thresholds

,threshold,alertas,percentual_alertado,precision,recall,f1,verdadeiros_positivos,falsos_positivos,falsos_negativos,verdadeiros_negativos
0,9.999194e-01,1,0.000514,1.000000,0.000799,0.001596,1,0,1251,193249
1,9.998375e-01,2,0.001028,1.000000,0.001597,0.003190,2,0,1250,193249
2,9.998334e-01,3,0.001542,1.000000,0.002396,0.004781,3,0,1249,193249
3,9.998287e-01,4,0.002057,1.000000,0.003195,0.006369,4,0,1248,193249
4,9.998132e-01,5,0.002571,1.000000,0.003994,0.007955,5,0,1247,193249
...,...,...,...,...,...,...,...,...,...,...
194266,1.315664e-07,194497,99.997943,0.006437,1.000000,0.012792,1252,193245,0,4
194267,1.188695e-07,194498,99.998458,0.006437,1.000000,0.012792,1252,193246,0,3
194268,1.069792e-07,194499,99.998972,0.006437,1.000000,0.012792,1252,193247,0,2
194269,8.436242e-08,194500,99.999486,0.006437,1.000000,0.012792,1252,193248,0,1


In [42]:
def avaliar_threshold_operacional(
    threshold
):

    previsto = (
        scores_validacao
        >= threshold
    )

    tp = int(
        (
            (previsto == 1)
            &
            (y_validacao == 1)
        ).sum()
    )

    fp = int(
        (
            (previsto == 1)
            &
            (y_validacao == 0)
        ).sum()
    )

    fn = int(
        (
            (previsto == 0)
            &
            (y_validacao == 1)
        ).sum()
    )

    tn = int(
        (
            (previsto == 0)
            &
            (y_validacao == 0)
        ).sum()
    )

    precision = (
        tp / (tp + fp)
        if tp + fp > 0
        else 0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn > 0
        else 0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall > 0
        else 0
    )

    alertas = tp + fp

    return {
        "threshold": threshold,
        "alertas": alertas,
        "percentual_alertado":
            alertas
            / len(y_validacao)
            * 100,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn
    }

In [44]:
thresholds_exemplo = [
    0.01,
    0.05,
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90,
    0.95,
    0.99
]

In [45]:
comparacao_thresholds = pd.DataFrame(
    [
        avaliar_threshold_operacional(
            threshold
        )
        for threshold
        in thresholds_exemplo
    ]
)

In [50]:
print(comparacao_thresholds[
    [
        "threshold",
        "alertas",
        "percentual_alertado",
        "precision",
        "recall",
        "f1",
        "tp",
        "fp",
        "fn"
    ]
])

    threshold  alertas  percentual_alertado  precision    recall        f1  \
0        0.01     3875             1.992278   0.313032  0.968850  0.473181   
1        0.05     1749             0.899224   0.662664  0.925719  0.772409   
2        0.10     1393             0.716192   0.801149  0.891374  0.843856   
3        0.20     1168             0.600511   0.909247  0.848243  0.877686   
4        0.30     1100             0.565550   0.943636  0.829073  0.882653   
5        0.40     1057             0.543442   0.957427  0.808307  0.876570   
6        0.50     1019             0.523905   0.965653  0.785942  0.866579   
7        0.60      984             0.505910   0.969512  0.761981  0.853309   
8        0.70      929             0.477633   0.975242  0.723642  0.830812   
9        0.80      854             0.439072   0.987119  0.673323  0.800570   
10       0.90      811             0.416964   0.995068  0.644569  0.782356   
11       0.95      757             0.389201   0.998679  0.603834

In [47]:
CAMINHO_TABELA_THRESHOLDS = (
    PASTA_TABELAS
    / "analise_thresholds_catboost_v1.csv"
)

In [48]:
tabela_thresholds.to_csv(
    CAMINHO_TABELA_THRESHOLDS,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Tabela salva em:",
    CAMINHO_TABELA_THRESHOLDS.resolve()
)

Tabela salva em: /Users/lucassantos/Documents/ccard_fraud_ml/outputs/tables/06_politica_decisao/analise_thresholds_catboost_v1.csv


In [49]:
comparacao_thresholds.to_csv(
    PASTA_TABELAS
    / "thresholds_referencia_catboost_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

In [51]:
def aplicar_politica(
    scores,
    threshold_revisao,
    threshold_critico
):

    scores = np.asarray(
        scores,
        dtype=float
    )

    decisoes = np.full(
        len(scores),
        "APROVAR",
        dtype=object
    )

    decisoes[
        scores >= threshold_revisao
    ] = "REVISAR"

    decisoes[
        scores >= threshold_critico
    ] = "ALERTA_CRITICO"

    return decisoes

In [52]:
def avaliar_politica(
    y_real,
    scores,
    threshold_revisao,
    threshold_critico
):

    if (
        threshold_revisao
        >= threshold_critico
    ):
        raise ValueError(
            "O threshold de revisão deve "
            "ser menor que o threshold crítico."
        )

    y_real = np.asarray(
        y_real,
        dtype=int
    )

    decisoes = aplicar_politica(
        scores=scores,
        threshold_revisao=threshold_revisao,
        threshold_critico=threshold_critico
    )

    resultados = []

    for decisao in [
        "APROVAR",
        "REVISAR",
        "ALERTA_CRITICO"
    ]:

        mascara = (
            decisoes == decisao
        )

        quantidade = int(
            mascara.sum()
        )

        fraudes = int(
            y_real[
                mascara
            ].sum()
        )

        legitimas = (
            quantidade
            - fraudes
        )

        taxa_fraude = (
            fraudes / quantidade
            if quantidade > 0
            else 0
        )

        resultados.append({
            "decisao":
                decisao,

            "quantidade":
                quantidade,

            "percentual_base":
                quantidade
                / len(y_real)
                * 100,

            "fraudes":
                fraudes,

            "legitimas":
                legitimas,

            "taxa_fraude":
                taxa_fraude
        })

    return pd.DataFrame(
        resultados
    )

In [53]:
politica_020_090 = (
    avaliar_politica(
        y_real=y_validacao,
        scores=scores_validacao,
        threshold_revisao=0.20,
        threshold_critico=0.90
    )
)

In [54]:
politicas_candidatas = [
    {
        "nome": "cobertura",
        "threshold_revisao": 0.10,
        "threshold_critico": 0.90
    },
    {
        "nome": "equilibrio",
        "threshold_revisao": 0.20,
        "threshold_critico": 0.90
    },
    {
        "nome": "seletiva",
        "threshold_revisao": 0.30,
        "threshold_critico": 0.90
    }
]

In [55]:
for politica in politicas_candidatas:

    print(
        "\n",
        "=" * 60
    )

    print(
        politica["nome"].upper()
    )

    display(
        avaliar_politica(
            y_real=y_validacao,
            scores=scores_validacao,
            threshold_revisao=(
                politica[
                    "threshold_revisao"
                ]
            ),
            threshold_critico=(
                politica[
                    "threshold_critico"
                ]
            )
        )
    )


COBERTURA


,decisao,quantidade,percentual_base,fraudes,legitimas,taxa_fraude
0,APROVAR,193108,99.283808,136,192972,0.000704
1,REVISAR,582,0.299227,309,273,0.530928
2,ALERTA_CRITICO,811,0.416964,807,4,0.995068



EQUILIBRIO


,decisao,quantidade,percentual_base,fraudes,legitimas,taxa_fraude
0,APROVAR,193333,99.399489,190,193143,0.000983
1,REVISAR,357,0.183547,255,102,0.714286
2,ALERTA_CRITICO,811,0.416964,807,4,0.995068



SELETIVA


,decisao,quantidade,percentual_base,fraudes,legitimas,taxa_fraude
0,APROVAR,193401,99.434450,214,193187,0.001107
1,REVISAR,289,0.148585,231,58,0.799308
2,ALERTA_CRITICO,811,0.416964,807,4,0.995068


In [57]:
for politica in politicas_candidatas:

    print(
        "\n",
        "=" * 60
    )

    print(
        politica["nome"].upper()
    )
    print(
    display(
        avaliar_politica(
            y_real=y_validacao,
            scores=scores_validacao,
            threshold_revisao=(
                politica[
                    "threshold_revisao"
                ]
            ),
            threshold_critico=(
                politica[
                    "threshold_critico"
                ]
            )
        )
    ))


COBERTURA


,decisao,quantidade,percentual_base,fraudes,legitimas,taxa_fraude
0,APROVAR,193108,99.283808,136,192972,0.000704
1,REVISAR,582,0.299227,309,273,0.530928
2,ALERTA_CRITICO,811,0.416964,807,4,0.995068


None

EQUILIBRIO


,decisao,quantidade,percentual_base,fraudes,legitimas,taxa_fraude
0,APROVAR,193333,99.399489,190,193143,0.000983
1,REVISAR,357,0.183547,255,102,0.714286
2,ALERTA_CRITICO,811,0.416964,807,4,0.995068


None

SELETIVA


,decisao,quantidade,percentual_base,fraudes,legitimas,taxa_fraude
0,APROVAR,193401,99.434450,214,193187,0.001107
1,REVISAR,289,0.148585,231,58,0.799308
2,ALERTA_CRITICO,811,0.416964,807,4,0.995068


None


In [59]:
def resumir_politica(
    nome,
    threshold_revisao,
    threshold_critico,
    y_real,
    scores
):

    resultado = avaliar_politica(
        y_real=y_real,
        scores=scores,
        threshold_revisao=threshold_revisao,
        threshold_critico=threshold_critico
    )

    aprovar = (
        resultado
        .loc[
            resultado["decisao"] == "APROVAR"
        ]
        .iloc[0]
    )

    revisar = (
        resultado
        .loc[
            resultado["decisao"] == "REVISAR"
        ]
        .iloc[0]
    )

    critico = (
        resultado
        .loc[
            resultado["decisao"] == "ALERTA_CRITICO"
        ]
        .iloc[0]
    )

    total_fraudes = int(
        np.asarray(y_real).sum()
    )

    fraudes_detectadas = int(
        revisar["fraudes"]
        +
        critico["fraudes"]
    )

    fraudes_perdidas = int(
        aprovar["fraudes"]
    )

    recall_global = (
        fraudes_detectadas
        / total_fraudes
    )

    total_encaminhado = int(
        revisar["quantidade"]
        +
        critico["quantidade"]
    )

    percentual_encaminhado = (
        total_encaminhado
        / len(y_real)
        * 100
    )

    return {
        "politica": nome,
        "threshold_revisao":
            threshold_revisao,
        "threshold_critico":
            threshold_critico,

        "fraudes_detectadas":
            fraudes_detectadas,

        "fraudes_perdidas":
            fraudes_perdidas,

        "recall_global":
            recall_global,

        "total_encaminhado":
            total_encaminhado,

        "percentual_encaminhado":
            percentual_encaminhado,

        "fila_revisao":
            int(revisar["quantidade"]),

        "fraudes_revisao":
            int(revisar["fraudes"]),

        "precision_revisao":
            float(revisar["taxa_fraude"]),

        "alertas_criticos":
            int(critico["quantidade"]),

        "fraudes_criticas":
            int(critico["fraudes"]),

        "precision_critico":
            float(critico["taxa_fraude"])
    }

In [60]:
resumo_politicas = pd.DataFrame([
    resumir_politica(
        nome="Cobertura",
        threshold_revisao=0.10,
        threshold_critico=0.90,
        y_real=y_validacao,
        scores=scores_validacao
    ),

    resumir_politica(
        nome="Equilíbrio",
        threshold_revisao=0.20,
        threshold_critico=0.90,
        y_real=y_validacao,
        scores=scores_validacao
    ),

    resumir_politica(
        nome="Seletiva",
        threshold_revisao=0.30,
        threshold_critico=0.90,
        y_real=y_validacao,
        scores=scores_validacao
    )
])

In [61]:
resumo_politicas.T

,0,1,2
politica,Cobertura,Equilíbrio,Seletiva
threshold_revisao,0.1,0.2,0.3
threshold_critico,0.9,0.9,0.9
fraudes_detectadas,1116,1062,1038
fraudes_perdidas,136,190,214
recall_global,0.891374,0.848243,0.829073
total_encaminhado,1393,1168,1100
percentual_encaminhado,0.716192,0.600511,0.56555
fila_revisao,582,357,289
fraudes_revisao,309,255,231


In [62]:
resumo_politicas.to_csv(
    PASTA_TABELAS
    / "comparacao_politicas_candidatas.csv",
    index=False,
    encoding="utf-8-sig"
)

## Política de decisão selecionada

Após a avaliação de diferentes configurações no conjunto de
validação, foi selecionada a política denominada **Equilíbrio**.

A política utiliza dois pontos de corte:

- score inferior a 0,20: **APROVAR**;
- score entre 0,20 e 0,90: **REVISAR**;
- score igual ou superior a 0,90: **ALERTA_CRÍTICO**.

A configuração apresentou Recall global de aproximadamente 84,82%,
identificando 1.062 das 1.252 fraudes presentes no conjunto de
validação.

Ao todo, 1.168 transações, correspondentes a aproximadamente 0,60%
da base de validação, foram encaminhadas para as faixas de revisão
ou alerta crítico.

A faixa REVISAR apresentou 357 transações, das quais 255 eram
fraudulentas, resultando em taxa de fraude de aproximadamente
71,43%.

A faixa ALERTA_CRÍTICO apresentou 811 transações, das quais 807 eram
fraudulentas, correspondendo a aproximadamente 99,51% de concentração
da classe positiva.

A política foi escolhida por representar um compromisso entre
cobertura das fraudes e volume operacional de análise. Uma política
mais permissiva, com threshold de revisão igual a 0,10, aumentaria o
Recall, porém elevaria significativamente o número de transações
encaminhadas para revisão. Já uma política mais seletiva, com
threshold igual a 0,30, reduziria relativamente pouco a fila de
revisão, ao custo de deixar uma quantidade adicional de fraudes sem
detecção.

Os thresholds foram definidos exclusivamente sobre o conjunto de
validação. O conjunto de teste não foi utilizado durante esse processo.

In [63]:
POLITICA_FINAL = {
    "policy_version": "decision_policy_v1",
    "model_version": "catboost_v1",

    "threshold_revisao": 0.20,
    "threshold_alerta_critico": 0.90,

    "decisoes": {
        "APROVAR": "score < 0.20",
        "REVISAR": "0.20 <= score < 0.90",
        "ALERTA_CRITICO": "score >= 0.90"
    },

    "selection_dataset": "validation",

    "validation_metrics": {
        "total_transacoes": 194501,
        "total_fraudes": 1252,

        "fraudes_detectadas": 1062,
        "fraudes_perdidas": 190,

        "recall_global": 0.848243,

        "total_encaminhado": 1168,
        "percentual_encaminhado": 0.600511,

        "revisao": {
            "quantidade": 357,
            "fraudes": 255,
            "legitimas": 102,
            "taxa_fraude": 0.714286
        },

        "alerta_critico": {
            "quantidade": 811,
            "fraudes": 807,
            "legitimas": 4,
            "taxa_fraude": 0.995068
        }
    }
}

In [65]:
import json
from pathlib import Path

In [66]:
CAMINHO_POLITICA = (
    PROJECT_ROOT
    / "models"
    / "catboost_v1"
    / "decision_policy.json"
)

with open(
    CAMINHO_POLITICA,
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(
        POLITICA_FINAL,
        arquivo,
        ensure_ascii=False,
        indent=4
    )

print(
    "Política salva em:",
    CAMINHO_POLITICA.resolve()
)

Política salva em: /Users/lucassantos/Documents/ccard_fraud_ml/models/catboost_v1/decision_policy.json
